# DataQuery AI

## install the package

In [112]:
# pip install -U  langchain-community

In [113]:
# pip install langchain_google_genai

In [114]:
# pip install python-docx

In [115]:
# pip install pypdf

In [116]:
# pip install chromadb

## Retrive LLM model using gemini

In [111]:
import os
gemini_api=os.environ.get('GEMINI_API_KEY')

In [84]:
import warnings
warnings.filterwarnings('ignore')

In [86]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    api_key=gemini_api,
    model="gemini-2.5-flash",
    temperature=0.7,
    max_tokens=None,

)

In [87]:
llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x7964508e90a0>, default_metadata=(), model_kwargs={})

## Load the pdf

In [88]:
from langchain_community.document_loaders import PyPDFLoader
from docx import Document

In [89]:
pdf_path=r'/Data.pdf'
loader=PyPDFLoader(pdf_path)
documents=loader.load()

## Split into Text

In [90]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [91]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

In [92]:
docs=splitter.split_documents(documents)

In [93]:
print(len(docs))

649


## Divide text into Embeddings and Store in vector database

In [94]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [95]:
embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [96]:
vectordb=Chroma.from_documents(
    documents=docs,
    embedding=embeddings
)
vectordb.persist()

In [97]:
retriever=vectordb.as_retriever(search_kwargs={'k':10})

## Create a PromptTemplate

In [98]:
from langchain_core.prompts import PromptTemplate

In [99]:
prompt_template=PromptTemplate(
    template="""
    you are a helpful **Data Scientist**.
    Explain data science concept short & simple way.

    Context rules:
    -Use only the provided CONTEXT chunks from the retriever.
    -Explain in a clear and professional tone.
    -Do not make up information beyond the context.
    -Do not tell As a Data Scientist in answer
    -if the answer is not availabe in the context, say "I don't know".

    CONTEXT:
    {context}

    QUESTION:
    {input}

    Answer as a Data Scientist
    """,
    input_variables=['context','input']
)

## Create Retrival Chain

In [100]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [101]:
chain_QA = create_stuff_documents_chain(llm, prompt_template)
Chain = create_retrieval_chain(retriever, chain_QA)


## Ask the Question

In [102]:
response = Chain.invoke({"input": "What is Linear Regression?"})

In [103]:
print(response["answer"])

Linear regression is a widely recognized modeling approach. It determines the relationship between dependent and independent variables by using a straight line, known as a regression line, and is typically expressed in equation form. This technique is applicable when the dependent variable has an unlimited range. However, if the dependent variable is discrete, a different type of model is required.


In [104]:
query='what is machine learning'
response = Chain.invoke({"input":query})
print("\n---model answer---\n")
print(response["answer"])



---model answer---

Machine learning is a method of data analysis that involves training systems or algorithms to gain information from a data set. Its central goal is to teach a computer to recognize patterns. This technique automates the building of analytical models, using algorithms that continuously adapt and learn from data and previous computations to find information without direct programming.


In [105]:
query='what is Big data'
response = Chain.invoke({"input":query})
print("\n---model answer---\n")
print(response["answer"])


---model answer---

Big data refers to datasets whose size is beyond the ability of typical database software tools to capture, store, manage, and analyze. The concept was defined by Doug Laney in terms of three key characteristics:

*   **Volume:** This refers to the sheer size of the dataset.
*   **Velocity:** This is the speed at which data is acquired and utilized, with companies aiming to derive meaning from data more quickly, often in real-time.
*   **Variety:** This relates to the many different types of data that can be used for collection and analysis, extending beyond what is typically found in a normal database.
